<p align="center"><img src="https://raw.githubusercontent.com/arab-future-academy/deep_dive_in_gen_ai/refs/heads/main/imgs/arabfutureacademy.png" alt="Arabic Future Academy" width="720" /></p>

# كومفي يو أي 

# **الإعدادات المركزية**

**هنا بتحدد إعدادات الجلسة: التخزين، حزمة الموديلات (`BUNDLE`)، وملفات إضافية (`EXTRA_FILES`).**

**التحميل من Hugging Face بمراجع مثبّتة (SHA) — يذهب لقرص `/content` وليس RAM.**

---

## ✅ تغطية القوالب الرسمية (ComfyUI Templates)

كل الحزم التالية مكتملة 100% — لا ينقصها أي موديل:

| القالب الرسمي | BUNDLE | الحجم | ملاحظة T4 |
|---|---|---|---|
| `image_qwen_image_edit_2511` | `qwen-edit-2511` | ~31 GB | ✅ (fp8mixed = القالب الافتراضي) |
| `image_qwen_image_2_1_image_edit` | `qwen-image-2.1-edit` | ~17 GB | ✅ |
| `video_wan21_scail2_character_replacement` | `wan-scail2-t4` | ~26 GB | ✅ (int8 official) |
| `video_wan_vace_inpainting` | `wan-vace-t4` | ~15 GB | ✅ (1.3B بديل 14B) |
| `3d_pixal3d_trellis2_image_to_model` | `3d-pixal3d-trellis2` | ~15 GB | ✅ |
| `llm_gemma4_text_gen` | `llm_gemma4_text_gen` | ~9 GB | ✅ |
| `llm_qwen3_5_text_gen` | `llm_qwen3_5_text_gen` | ~9 GB | ✅ |

### حزم إضافية (صور / صوت / أدوات)
| BUNDLE | الاستخدام | الحجم |
|---|---|---|
| `z-image-turbo-t4` | توليد صور سريع (int8) | ~12 GB |
| `z-image-turbo` | توليد صور bf16 | ~21 GB |
| `sd15-classic` | SD1.5 + CLIP + ControlNet | ~8 GB |
| `flux2-klein` | Flux2 Klein | ~9 GB |
| `chatterbox-tts` | نص → كلام | ~11 GB |
| `depth-3d-trellis` | عمق / 3D | ~5 GB |
| `extras` | مساعدة اختيارية | ~16 GB |
| `starter-t4` | صور سريعة + Qwen3.5-2B | ~17 GB |

### LLM — GGUF (لعقد ComfyUI-GGUF / LLM-text-processor)
| BUNDLE | الحجم |
|---|---|
| `llm-qwen3-8b-t4` | ~5 GB |
| `llm-qwen2.5-7b-t4` | ~5 GB |
| `llm-qwen2.5-3b-t4` | ~2 GB |
| `llm-qwen2.5-1.5b-t4` | ~1 GB |

### حزم للعتاد الأعلى (ليس T4 مجاني)
| BUNDLE | الحجم | التحذير |
|---|---|---|
| `wan-scail2-full` | ~27 GB | int8 + LoRA كاملة |
| `wan-vace-full` | ~45 GB | 14B fp16 — **لا يسع T4** |

---

## 📁 تغطية مجلدات ComfyUI (folder_paths.py)

الكتالوج يغطي هذه المجلدات المهمة:

| المجلد | الحالة | ملاحظة |
|---|---|---|
| `checkpoints` | ✅ | SD1.5, SAM3.1 |
| `clip` | ✅ | CLIP-L |
| `clip_vision` | ✅ | DINO-v3, CLIP-H |
| `controlnet` | ✅ | Canny SD1.5 |
| `diffusion_models` | ✅ | Qwen, Flux, Wan, Z-Image |
| `geometry_estimation` | ✅ | MoGe-2 |
| `loras` | ✅ | Lightning, SCAIL-2, CausVid |
| `text_encoders` | ✅ | Qwen-VL, UMT5, Gemma4, Qwen3.5 |
| `upscale_models` | ✅ | 4x-UltraSharp |
| `vae` | ✅ | Qwen, Flux2, Wan, SD1.5 |
| `vae_approx` | ✅ | للعرض السريع |

**ملاحظة:** مجلدات مثل `audio_encoders`, `configs`, `detection`, `diffusers`, `embeddings`, `frame_interpolation`, `gligen`, `hypernetworks`, `latent_upscale_models`, `model_patches`, `optical_flow`, `photomaker`, `style_models`, `unet` تُنشأ تلقائياً عند أول تشغيل ولا تحتاج موديلات مسبقاً — يمكنك إضافتها لاحقاً عبر `EXTRA_FILES`.

---

## 💡 نصائح
1. **حزمة واحدة كبيرة في الجلسة** (قرص Colab ~78GB).
2. `HF_XET_HIGH_PERF = False` يمنع انهيار RAM.
3. `EXTRA_FILES` لإضافة ملفات من الكتالوج أو `hf://repo/path`.
4. **تثبيت الفراشة:** أي تعديل على الموديلات ينعكس على preflight وself-heal — راجع التبعات قبل أي تغيير.


In [ ]:
#@title 0. Central Configuration
#@markdown Set the main options here before running the rest of the notebook.

# ── Model selection (three-tier flexibility) ──────────────────────
#@markdown **Model bundle:** curated sets from the pinned manifest. `none` = no automatic model pull.
BUNDLE = "none" #@param ["none", "z-image-turbo-t4", "z-image-turbo", "sd15-classic", "qwen-edit-2511", "qwen-image-2.1-edit", "flux2-klein", "chatterbox-tts", "depth-3d-trellis", "3d-pixal3d-trellis2", "wan-vace-t4", "wan-scail2-t4", "extras", "llm-qwen3-8b-t4", "llm-qwen2.5-7b-t4", "llm-qwen2.5-3b-t4", "llm-qwen2.5-1.5b-t4", "llm_gemma4_text_gen", "llm_gemma4_text_gen_t4", "llm_qwen3_5_text_gen", "llm_qwen3_5_text_gen_t4", "starter-t4", "wan-scail2-full", "wan-vace-full"]
#@markdown **Extra files:** one per line. Each line is either a `dest` path from the manifest, or a full `hf://repo_id/path/file` reference.
EXTRA_FILES = "" #@param {type:"string"}
#@markdown **Boot cache source:** `hf` = fast pull from our Hugging Face repo · `drive` = Google Drive tarball · `none` = cold build.
CACHE_SOURCE = "hf" #@param ["hf", "drive", "none"]
#@markdown **Manifest URL** (pinned catalog of verified model sources).
MANIFEST_URL = "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml" #@param {type:"string"}
#@markdown **Xet high-performance transfers** (faster Hugging Face downloads).
HF_XET_HIGH_PERF = False #@param {type:"boolean"}  # False = classic HTTP stream (safe for free Colab RAM)
# RAM tip: free Colab has ~12GB system RAM. Download staging uses /content (disk), never /tmp (RAM).
# Large bundles (qwen-edit-2511 32GB, wan-scail2 26GB) need disk headroom — one big bundle per session.

# ── Storage and updates ──
MOUNT_DRIVE = True #@param {type:"boolean"}
FORCE_REMOUNT_DRIVE = False #@param {type:"boolean"}
UPDATE_COMFY_UI = True #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}

# ── Paths ──
LOCAL_WORKSPACE = "/content/ComfyUI" #@param {type:"string"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI" #@param {type:"string"}

# ── Runtime checks ──
RUN_HEALTH_CHECKS = True #@param {type:"boolean"}
MIN_FREE_DISK_GB = 15 #@param {type:"integer"}

CACHE_TAR = f"{DRIVE_WORKSPACE}/comfy_ui_cache.tar"
HF_BOOT_CACHE_REPO = "awAyham/comfyui-boot-cache"
HF_BUNDLE_REPO = "awAyham/comfyui-models"

print("Configuration loaded.")
_n_extra = len([l for l in EXTRA_FILES.splitlines() if l.strip() and not l.strip().startswith("#")])
print(f"  Bundle: {BUNDLE} · Extra files: {_n_extra} · Cache: {CACHE_SOURCE}")
print(f"  Local workspace: {LOCAL_WORKSPACE}")
print(f"  Drive workspace: {DRIVE_WORKSPACE}")


# **تهيئة النظام**

**في الخلية دي بنجهّز البيئة كلها: بنركّب جوجل درايف، بننزّل ComfyUI ونحدّثه، وبننصّب كل المكتبات الأساسية اللي البرنامج محتاجها.**

**خليها تخلّص كامل قبل ما تكمل للخلية اللي بعدها، وراقب الرسائل اللي بتطلع — لو ظهر خطأ أحمر وقّف واقرأ الرسالة.**

In [ ]:
#@title 1. System Initialization
#@markdown **Run this cell first.** <br>
#@markdown Mounts Drive, sets up ComfyUI, and installs core dependencies. (Boot-cache restore moved to cell 2c.)

import os
import shutil
import subprocess
import sys
from google.colab import drive

# Defaults allow this cell to run even if the config cell was skipped.
MOUNT_DRIVE = globals().get("MOUNT_DRIVE", True)
FORCE_REMOUNT_DRIVE = globals().get("FORCE_REMOUNT_DRIVE", False)
UPDATE_COMFY_UI = globals().get("UPDATE_COMFY_UI", True)
INSTALL_COMFYUI_MANAGER = globals().get("INSTALL_COMFYUI_MANAGER", True)
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ComfyUI")
RUN_HEALTH_CHECKS = globals().get("RUN_HEALTH_CHECKS", True)
MIN_FREE_DISK_GB = globals().get("MIN_FREE_DISK_GB", 15)
CACHE_TAR = globals().get("CACHE_TAR", os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar"))


# ── CRITICAL: keep ALL temp + HF cache on /content disk (not /tmp RAM) ──
os.environ["TMPDIR"] = "/content/tmp"
os.environ["TEMP"] = "/content/tmp"
os.environ["TMP"] = "/content/tmp"
os.makedirs("/content/tmp", exist_ok=True)
os.environ.setdefault("HF_HOME", "/content/hf_home")
os.makedirs("/content/hf_home", exist_ok=True)
# Disable Xet reconstruction by default (RAM spike on free Colab)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
print(f"[ENV] TMPDIR={os.environ['TMPDIR']} · HF_HOME={os.environ.get('HF_HOME')} · XET disabled by default")

print("[SYSTEM] Initialization started.\n")

def stream_cmd(cmd, cwd=None):
    """Streams shell commands directly to the output"""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()

def run_health_checks(stage):
    if not RUN_HEALTH_CHECKS:
        return

    print(f"\n[HEALTH] {stage}")

    try:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
        if gpu.returncode == 0 and gpu.stdout.strip():
            print(f"[OK] GPU: {gpu.stdout.strip()}")
        else:
            print("[WARN] nvidia-smi returned no GPU. In Colab: Runtime > Change runtime type > GPU (T4).")
    except FileNotFoundError:
        print("[WARN] nvidia-smi not found — no NVIDIA GPU. In Colab: Runtime > Change runtime type > GPU (T4).")

    total, used, free = shutil.disk_usage("/content")
    free_gb = free / (1024 ** 3)
    print(f"[OK] /content free disk: {free_gb:.1f} GB")
    if free_gb < MIN_FREE_DISK_GB:
        print(f"[WARN] Free disk is below {MIN_FREE_DISK_GB} GB. Large model downloads may fail.")

    if MOUNT_DRIVE:
        drive_root = "/content/drive/MyDrive"
        if os.path.exists(drive_root):
            print(f"[OK] Google Drive mounted: {drive_root}")
        else:
            print("[INFO] Google Drive is not mounted yet.")

    if os.path.exists(LOCAL_WORKSPACE):
        print(f"[OK] Local workspace exists: {LOCAL_WORKSPACE}")
    else:
        print(f"[INFO] Local workspace will be created: {LOCAL_WORKSPACE}")

def mount_google_drive():
    drive_root = "/content/drive/MyDrive"
    if os.path.exists(drive_root) and not FORCE_REMOUNT_DRIVE:
        print(f"[OK] Google Drive already mounted: {drive_root}")
        return

    try:
        drive.mount('/content/drive', force_remount=FORCE_REMOUNT_DRIVE)
    except Exception as err:
        if not FORCE_REMOUNT_DRIVE:
            print("[DRIVE] First mount failed. Retrying once with force_remount=True...")
            try:
                drive.mount('/content/drive', force_remount=True)
            except Exception as retry_err:
                raise RuntimeError(
                    "Google Drive mount failed. In Colab, try Runtime > Restart runtime, "
                    "then run the configuration and initialization cells again. Also check that "
                    "browser popups/cookies are allowed for colab.research.google.com. "
                    f"Original error: {err}; retry error: {retry_err}"
                )
        else:
            raise RuntimeError(
                "Google Drive mount failed even with force_remount enabled. Try Runtime > Restart runtime, "
                "then run the cells again and complete the Google authorization popup. "
                f"Original error: {err}"
            )

    if not os.path.exists(drive_root):
        raise RuntimeError("Google Drive mount finished, but /content/drive/MyDrive was not found.")

run_health_checks("Before setup")

# 1. Mount Google Drive
if MOUNT_DRIVE:
    print("[DRIVE] Mounting Google Drive...")
    mount_google_drive()
    os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
    run_health_checks("After Drive mount")

# 2. Setup ComfyUI Core
if not os.path.exists(LOCAL_WORKSPACE):
    print("[COMFYUI] Cloning ComfyUI...")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
else:
    if UPDATE_COMFY_UI:
        print("[COMFYUI] Checking for updates...")
        subprocess.run(["git", "pull"], cwd=LOCAL_WORKSPACE)

# 3. Configure Hybrid Storage & Tarball Cache
heavy_dirs = ["models", "output", "input"]
print("\n[STORAGE] Routing heavy model directories directly to Drive...")
for d in heavy_dirs:
    local_path = os.path.join(LOCAL_WORKSPACE, d)
    drive_path = os.path.join(DRIVE_WORKSPACE, d)
    if not os.path.exists(drive_path): os.makedirs(drive_path, exist_ok=True)
    if os.path.exists(local_path) and not os.path.islink(local_path): shutil.rmtree(local_path)
    if not os.path.exists(local_path): os.symlink(drive_path, local_path)

print("\n[CACHE] Boot-cache restore moved to cell 2c (hf | drive | none).")
os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_WORKSPACE, "user"), exist_ok=True)

# 4. Install ComfyUI Manager
if INSTALL_COMFYUI_MANAGER:
    manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
    if not os.path.exists(manager_path):
        print("\n[MANAGER] Installing ComfyUI Manager...")
        subprocess.run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager.git", manager_path])
    else:
        print("\n[OK] ComfyUI Manager verified.")
        subprocess.run(["git", "pull"], cwd=manager_path)

# 5. Core Python Dependencies
print("\n[PYTHON] Installing base Python requirements...")
stream_cmd(["pip", "install", "xformers!=0.0.18", "-r", "requirements.txt", "--extra-index-url", "https://download.pytorch.org/whl/cu121"], cwd=LOCAL_WORKSPACE)
stream_cmd(["pip", "install", "insightface", "onnxruntime-gpu"])
stream_cmd(["pip", "install", "-q", "-U", "huggingface_hub", "pyyaml"])
# Colab torch builds change often; mismatched xformers wheels only add warnings.
# PyTorch SDPA is the reliable fallback for ComfyUI attention.
stream_cmd(["pip", "uninstall", "-y", "xformers"])

# 6. AUTO-HEAL: Custom Node Dependencies
print("\n[AUTO-HEAL] Scanning local custom nodes for missing dependencies...")
custom_nodes_dir = os.path.join(LOCAL_WORKSPACE, "custom_nodes")
if os.path.exists(custom_nodes_dir):
    for item in os.listdir(custom_nodes_dir):
        node_path = os.path.join(custom_nodes_dir, item)
        req_file = os.path.join(node_path, "requirements.txt")
        if os.path.isdir(node_path) and os.path.exists(req_file):
            print(f"\n   Restoring dependencies for: {item}")
            stream_cmd(["pip", "install", "-r", "requirements.txt"], cwd=node_path)

# 7. Pip cache cleanup (frees disk for model downloads)
print("\n[PIP] Purging pip cache...")
os.system("pip cache purge > /dev/null 2>&1")

run_health_checks("After setup")
print("\n[SYSTEM READY] Initialization complete.")

# **تسجيل الدخول لهافينج فيس**

**في الخلية دي بنسجّل الدخول إلى Hugging Face باستخدام توكن محفوظ داخل أسرار كولاب باسم `HF_TOKEN` بالظبط.**

**التوكن ده محتاجه البرنامج عشان يسحب كاش الإقلاع الخاص (وأي موديل خاص). التوكن ما يتكتبش أبداً داخل النوتبوك مباشرة — دايماً من الأسرار.**

**لو ما عندكش توكن: غيّر `CACHE_SOURCE` في الإعدادات إلى `drive` أو `none`، وهتشتغل عادي مع الموديلات العامة.**

In [ ]:
#@title 1b. Hugging Face Login from Colab Secrets

import os
from google.colab import userdata
from huggingface_hub import login, whoami

CACHE_SOURCE = globals().get("CACHE_SOURCE", "hf")

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"[HF] Colab Secrets unavailable: {e}")

if HF_TOKEN is None:
    if CACHE_SOURCE == "hf":
        raise ValueError(
            "HF_TOKEN مطلوب: CACHE_SOURCE=hf بيحتاج توكن للوصول للكاش الخاص (awAyham/comfyui-boot-cache). "
            "أضف سر باسم HF_TOKEN في Colab (أيقونة المفاتيح على اليسار) وفعّل access للنتبوك، "
            "أو غيّر CACHE_SOURCE إلى drive/none في خلية الإعدادات."
        )
    print("[HF][WARN] No HF_TOKEN — public pulls will work but may hit rate limits. Private boot cache unavailable.")

if HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip().replace('"', '').replace("'", "")
    if HF_TOKEN.lower().startswith("bearer "):
        HF_TOKEN = HF_TOKEN.split(" ", 1)[1].strip()
    if not HF_TOKEN.startswith("hf_"):
        raise ValueError("التوكن لا يبدأ بـ hf_ — تأكد إن Secret اسمه HF_TOKEN وقيمته صحيحة.")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=True)
    info = whoami(token=HF_TOKEN)
    print("Logged in as:", info["name"])

# **تحميل الموديلات**

**بتحمّل `BUNDLE` + `EXTRA_FILES` من Hugging Face بمراجع مثبّتة، على قرص `/content` (مش RAM).**

### ✅ للقوالب الرسمية — كل الموديلات موجودة
- `qwen-edit-2511` → fp8mixed (القالب الافتراضي) + VL + VAE + Lightning LoRA + 4x-UltraSharp
- `wan-scail2-t4` → SCAIL-2 int8 + UMT5 + VAE + CLIP-H + DPO LoRA + SAM3.1
- `wan-vace-t4` → VACE 1.3B + UMT5 + VAE + CLIP-H + SAM3.1 + CausVid LoRA
- `3d-pixal3d-trellis2` → TRELLIS.2 + Pixal3D + DINO + MoGe + BiRefNet + VAEs
- `llm_gemma4_text_gen` / `llm_qwen3_5_text_gen` → نماذج TextGenerate الرسمية

### أثناء التحميل
- لو حصل OOM أو `reconstructing` طويل: أكّد `HF_XET_HIGH_PERF = False` وأعد التشغيل.
- في الآخر يظهر **[PULL REPORT]**.

### لو BUNDLE=none
- الخلية تُتخطى وتستخدم الروابط المباشرة في الخلية التالية.


In [ ]:
#@title 2. Model Pull (manifest-driven)
#@markdown Pulls the selected bundle + extra files from their pinned upstream sources (or our archive repo).

import os, sys, shutil, tempfile
import requests, yaml
from huggingface_hub import hf_hub_download, HfApi

BUNDLE = globals().get("BUNDLE", "none")
EXTRA_FILES = globals().get("EXTRA_FILES", "")
MANIFEST_URL = globals().get("MANIFEST_URL", "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml")
HF_XET_HIGH_PERF = globals().get("HF_XET_HIGH_PERF", False)  # False: stream HTTP, no Xet RAM spike
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
MIN_FREE_DISK_GB = globals().get("MIN_FREE_DISK_GB", 15)
HF_BUNDLE_REPO = globals().get("HF_BUNDLE_REPO", "awAyham/comfyui-models")

# Xet "reconstructing file" can spike RAM on free Colab (~12GB). Classic HTTP streams to disk.
# Set HF_XET_HIGH_PERF=True in Config only if you have >32GB RAM.
if HF_XET_HIGH_PERF:
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
else:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

MODELS_ROOT = os.path.join(LOCAL_WORKSPACE, "models")

# 1) Fetch pinned manifest (HEAD then GET — fail fast)
print(f"[MANIFEST] Fetching {MANIFEST_URL}")
head = requests.head(MANIFEST_URL, timeout=20, allow_redirects=True)
head.raise_for_status()
manifest = yaml.safe_load(requests.get(MANIFEST_URL, timeout=30).text)
FILES_BY_DEST = {f["dest"]: f for f in manifest.get("files", [])}
BUNDLES = manifest.get("bundles", {})
print(f"[MANIFEST] {len(FILES_BY_DEST)} files · bundles: {', '.join(BUNDLES)}")

# 2) Resolve selection: bundle (tier 1) + extra dest keys / hf:// refs (tier 2)
selected = []

if BUNDLE and BUNDLE != "none":
    if BUNDLE not in BUNDLES:
        raise SystemExit(f"[PULL] Unknown bundle '{BUNDLE}'. Available: {list(BUNDLES)}")
    for dest in BUNDLES[BUNDLE]:
        if dest not in FILES_BY_DEST:
            raise SystemExit(f"[PULL] Bundle '{BUNDLE}' references missing manifest entry: {dest}")
        selected.append(dict(FILES_BY_DEST[dest]))

for raw in (EXTRA_FILES or "").replace(",", "\n").split("\n"):
    spec = raw.strip()
    if not spec or spec.startswith("#"):
        continue
    if spec in FILES_BY_DEST:
        selected.append(dict(FILES_BY_DEST[spec]))
        continue
    if spec.startswith("hf://"):
        ref = spec[len("hf://"):]
        repo_id, _, remote = ref.partition("/")
        if not remote:
            raise SystemExit(f"[PULL] Bad hf:// reference (need hf://repo_id/path/file): {spec}")
        print(f"[PULL] EXTRA unpinned hf:// ref — resolving HEAD revision for {repo_id} ...")
        sha = HfApi().model_info(repo_id).sha
        selected.append({"dest": f"extras/{os.path.basename(remote)}", "source": f"hf://{repo_id}",
                         "remote": remote, "revision": sha, "size_gb": None})
        continue
    raise SystemExit(f"[PULL] EXTRA_FILES entry not recognized (manifest dest or hf://repo/file): {spec}")

# de-dupe by dest (first wins)
deduped, seen = [], set()
for j in selected:
    if j["dest"] in seen:
        print(f"[PULL] skip duplicate: {j['dest']}")
        continue
    seen.add(j["dest"])
    deduped.append(j)
selected = deduped

if not selected:
    print("[PULL] Nothing selected (BUNDLE=none and EXTRA_FILES empty). Skipping model pull.")
else:
    # 3) Disk check on known sizes
    need_gb = sum(j["size_gb"] for j in selected if j.get("size_gb"))
    free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
    print(f"[DISK] Need ~{need_gb:.1f} GB (known sizes) · free on /content: {free_gb:.1f} GB")
    if free_gb - need_gb < MIN_FREE_DISK_GB:
        raise SystemExit(f"[DISK] Not enough free space: need {need_gb:.1f} GB + {MIN_FREE_DISK_GB} GB margin, have {free_gb:.1f} GB. Pick a smaller bundle.")

    # 4) Pull each entry
    results = []
    for j in selected:
        src, dest = j["source"], j["dest"]
        final_path = os.path.join(MODELS_ROOT, dest)
        if os.path.exists(final_path):
            size = os.path.getsize(final_path)
            expect = j.get("size_gb")
            if expect is None or abs(size - expect * 1e9) / max(size, 1) <= 0.02:
                print(f"   [SKIP] {dest} (exists, {size / 1e9:.2f} GB)")
                results.append((dest, "skip", size))
                continue
        if src.startswith("hf://") or src.startswith("archive://"):
            repo_id = src.split("://", 1)[1]
            remote = j.get("remote", dest)  # archive:// files live at their dest path
            rev = j.get("revision") or None
            tag = f"@{rev[:10]}" if rev else "@main (unpinned!)"
            print(f"   [HF] {repo_id} :: {remote} {tag}")
            # CRITICAL: /tmp is tmpfs (RAM) on Colab — never download there.
            # Stage under /content (disk) then move.
            _stage_root = "/content/.model_pull_tmp"
            os.makedirs(_stage_root, exist_ok=True)
            with tempfile.TemporaryDirectory(dir=_stage_root) as td:
                p = hf_hub_download(repo_id=repo_id, filename=remote, revision=rev,
                                    local_dir=td, repo_type="model")
                os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
                shutil.move(p, final_path)
            try:
                os.rmdir(_stage_root)
            except OSError:
                pass
            size = os.path.getsize(final_path)
            print(f"       -> {dest} ({size / 1e9:.2f} GB)")
            results.append((dest, "ok", size))
        elif src.startswith(("url://", "http://", "https://")):
            url = src[len("url://"):] if src.startswith("url://") else src
            print(f"   [URL] {url}")
            os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(final_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
            size = os.path.getsize(final_path)
            results.append((dest, "ok", size))
        else:
            print(f"   [ERR] unknown source scheme: {src}")
            results.append((dest, "err", 0))

    # 5) Report
    print("\n[PULL REPORT]")
    ok = sum(1 for _, s_, _ in results if s_ in ("ok", "skip"))
    bad = sum(1 for _, s_, _ in results if s_ == "err")
    for d, s_, sz in results:
        print(f"   {s_.upper():4}  {sz / 1e9:6.2f} GB  {d}")
    print(f"[PULL] done: {ok} ok/skipped · {bad} failed")

# **رفع موديل إلى الأرشيف (اختياري)**

**في الخلية دي بنرفع أي موديل نزّلناه من مصدر غير هافينج فيس (زي سيفيتاي) إلى مستودعنا الخاص على Hugging Face، عشان يبقى متاح في الجلسات الجاية بسرعة وثبات ومن غير ما الرابط الأصلي يضيع.**

**الخلية دي اختيارية: لو `ARCHIVE_PUSH_PATH` فاضية هتتخطى لوحدها. مثال على القيمة: `diffusion_models/my_model.safetensors` (المسار جوه مجلد `models/`).**

In [ ]:
#@title 2b. Archive Push (optional)
#@markdown Push a non-HF-sourced model into our archive repo (awAyham/comfyui-models) for future sessions.

import os, sys
from huggingface_hub import HfApi

ARCHIVE_PUSH_PATH = "" #@param {type:"string"}  # e.g. diffusion_models/my_civitai_model.safetensors (relative to models/)
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
HF_BUNDLE_REPO = globals().get("HF_BUNDLE_REPO", "awAyham/comfyui-models")

if not ARCHIVE_PUSH_PATH.strip():
    print("[ARCHIVE] ARCHIVE_PUSH_PATH empty — nothing to push (normal no-op).")
else:
    rel = ARCHIVE_PUSH_PATH.strip().lstrip("/")
    local = os.path.join(LOCAL_WORKSPACE, "models", rel)
    if not os.path.exists(local):
        raise SystemExit(f"[ARCHIVE] File not found: {local}")
    size_gb = os.path.getsize(local) / 1e9
    print(f"[ARCHIVE] Uploading {rel} ({size_gb:.2f} GB) -> {HF_BUNDLE_REPO} ...")
    HfApi().upload_file(path_or_fileobj=local, path_in_repo=rel, repo_id=HF_BUNDLE_REPO,
                        repo_type="model", commit_message=f"Add {rel}")
    print(f"[ARCHIVE] Done: https://huggingface.co/{HF_BUNDLE_REPO}/resolve/main/{rel}")

# **كاش الإقلاع**

**في الخلية دي بنسترجع كاش سريع (العقد `custom_nodes` + الإعدادات `user`) من هافينج فيس أو من جوجل درايف، عشان الجلسة الجديدة تفتح بسرعة من غير إعادة تنصيب كل حاجة.**

**أول مرة ممكن مفيش كاش خالص — ساعتها بنجهّل مجلدات جديدة عادي وكمان بننصّت متطلبات العقد اللي حضرتها. بعد ما تخلّص جلسة ناجحة، شغّل خلية حفظ الكاش في الآخر عشان الجلسة الجاية تفتح فوراً.**

In [ ]:
#@title 2c. Boot Cache Restore
#@markdown Restores the custom_nodes + user tarball according to CACHE_SOURCE (hf | drive | none).

import os, subprocess, sys
from huggingface_hub import hf_hub_download

CACHE_SOURCE = globals().get("CACHE_SOURCE", "hf")
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ComfyUI")
CACHE_TAR = globals().get("CACHE_TAR", os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar"))
HF_BOOT_CACHE_REPO = globals().get("HF_BOOT_CACHE_REPO", "awAyham/comfyui-boot-cache")

custom_nodes_dir = os.path.join(LOCAL_WORKSPACE, "custom_nodes")
user_dir = os.path.join(LOCAL_WORKSPACE, "user")

tar_path = None
if CACHE_SOURCE == "hf":
    print(f"[CACHE] Trying HF boot cache: {HF_BOOT_CACHE_REPO}/comfy_ui_cache.tar")
    try:
        # Stage under /content — default HF cache path is fine but keep TMPDIR safe
        os.makedirs("/content/.boot_cache_tmp", exist_ok=True)
        tar_path = hf_hub_download(HF_BOOT_CACHE_REPO, "comfy_ui_cache.tar", repo_type="model",
                                   local_dir="/content/.boot_cache_tmp")
        tar_path = os.path.join("/content/.boot_cache_tmp", "comfy_ui_cache.tar")
    except Exception as e:
        print(f"[CACHE] No HF cache yet ({type(e).__name__}) — first session? Building fresh dirs.")
elif CACHE_SOURCE == "drive":
    if os.path.exists(CACHE_TAR):
        tar_path = CACHE_TAR
        print(f"[CACHE] Using Drive cache: {CACHE_TAR}")
    else:
        print("[CACHE] No Drive cache found — building fresh dirs.")
else:
    print("[CACHE] CACHE_SOURCE=none — skipping restore (cold build).")

if tar_path:
    print("[CACHE] Extracting to local SSD ...")
    rc = os.system(f"tar -xf '{tar_path}' -C '{LOCAL_WORKSPACE}' > /dev/null 2>&1")
    if rc != 0:
        print("[CACHE][WARN] tar extraction returned non-zero — continuing with what was restored.")

os.makedirs(custom_nodes_dir, exist_ok=True)
os.makedirs(user_dir, exist_ok=True)

# Auto-heal node requirements AFTER restore (nodes may have arrived with the cache)
print("[CACHE] Auto-healing custom-node requirements ...")
for item in os.listdir(custom_nodes_dir):
    node_path = os.path.join(custom_nodes_dir, item)
    req = os.path.join(node_path, "requirements.txt")
    if os.path.isdir(node_path) and os.path.exists(req):
        print(f"   pip install -r requirements.txt  ({item})")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=node_path)

# Node requirements may have pulled xformers again — remove to keep SDPA path clean
os.system("pip uninstall -y xformers > /dev/null 2>&1")
print("[CACHE] Boot cache step complete.")

# **تحميل روابط مباشرة**

**في الخلية دي بنزّل موديلات أو عقد custom nodes من روابط مباشرة (زي سيفيتاي أو روابط جيت هاب). الصق الرابط في الخانة بتاعة الفئة الصح وشغّل الخلية.**

**لو الموديل اللي عايزه موجود في الكتالوج، الأفضل تستخدم `BUNDLE` أو `EXTRA_FILES` في الإعدادات — أسرع وأضمن.**

In [ ]:
#@title 3. Model & Node Downloader (direct URLs)
#@markdown Paste download links below. This supports **Checkpoints, Diffusion Models (Flux/UNET), Text Encoders (CLIP/T5), CLIP Vision, VAEs, LoRAs, and ControlNets**.

import os
import requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
CHECKPOINT_URLS = "" #@param {type:"string"}
UNET_DIFFUSION_URLS = "" #@param {type:"string"}
TEXT_ENCODER_URLS = "" #@param {type:"string"}
CLIP_VISION_URLS = "" #@param {type:"string"}
VAE_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "" #@param {type:"string"}

# --- Downloader Logic ---
DIRS = {
    "checkpoints":    os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":    os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
    "controlnet":     os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models": os.path.join(WORKSPACE, "models/upscale_models"),
    "embeddings":     os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":   os.path.join(WORKSPACE, "custom_nodes")
}

def get_filename(url, response):
    """Smartly determines filename from Content-Disposition or URL."""
    if "Content-Disposition" in response.headers:
        import re
        fname = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if fname: return fname[0]
    return unquote(os.path.basename(urlparse(url).path))

def download_file(url, target_dir):
    try:
        # Stream the download to get headers first
        response = requests.get(url, stream=True, allow_redirects=True)
        response.raise_for_status()

        filename = get_filename(url, response)
        file_path = os.path.join(target_dir, filename)
        total_size = int(response.headers.get('content-length', 0))

        if os.path.exists(file_path):
            print(f"   [SKIP] Exists: {filename}")
            return

        # Modern Progress Bar Log
        print(f"   [DOWNLOAD] {filename}")

        # The Progress Bar (Auto-Stretching)
        with tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc="      Progress",
            dynamic_ncols=True
        ) as bar:
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024): # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        print("      [OK] Download complete\n")

    except Exception as e:
        print(f"   [ERROR] Failed to download: {url}")
        print(f"      Error: {e}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return

    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    if not os.path.exists(target_dir): os.makedirs(target_dir, exist_ok=True)

    print(f"[CATEGORY] {os.path.basename(target_dir)}")

    for url in url_list:
        if is_node:
            node_name = url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if not os.path.exists(node_path):
                print(f"   [CLONE] Node: {node_name}...")
                !git clone {url} {node_path}
                # Auto-install requirements
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      Installing requirements...")
                    !pip install -r "{req}"
                print("      [OK] Installed\n")
            else:
                print(f"   [SKIP] Node exists: {node_name}\n")
        else:
            download_file(url, target_dir)

# --- Execution ---
process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("All tasks finished.")

# **حماية الجلسة من الانقطاع**

**في الخلية دي بنشغّل صوت صامت في الخلفية عشان المتصفح ما ينامش والجلسة ما تفصلش وانت شغال. شغّلها مرة واحدة بعد ما تفتح النفق.**

In [ ]:
# @title 3b. Session Anti-Disconnect
%%html
<b>Keep-Alive Audio</b><br>
<i>Running this silent audio loop prevents the browser tab from sleeping.</i><br>
<audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="width: 300px;" />

# **تشغيل كومفي يو**

**في الخلية دي بنجهّز ونطلق ComfyUI في الخلفية، وبنفتح نفق Cloudflare آمن، وبستنى لحد ما الرابط يجهز ويطبعه بوضوح — وبعدين الخلية بترجع فوراً عشان تقدر تشغّل أي خلية بعدها.**

** استنى لحد ما يظهر `ACTIVE TUNNEL: https://...trycloudflare.com` — افتحه في تبويب جديد. لو هتشتغل موديلات ضخمة (Z-Image / Qwen / Flux) اختار `Low VRAM` فوق قبل التشغيل.**

In [ ]:
#@title 4. Start ComfyUI Session
#@markdown Boots ComfyUI in the background, opens a Cloudflare tunnel, and prints the public URL. The cell returns once the tunnel is ready.

import subprocess, threading, time, socket, os, sys, re

# --- Session Configuration ---
#@markdown **Performance Profile:**
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]
#@markdown **Visual Settings:**
LIVE_GENERATION_PREVIEWS = True #@param {type:"boolean"}

WORKSPACE = "/content/ComfyUI"
ARGS = []
if "Low VRAM" in MEMORY_PROFILE:
    ARGS.append("--lowvram")
elif "High VRAM" in MEMORY_PROFILE:
    ARGS.append("--highvram")
if LIVE_GENERATION_PREVIEWS:
    ARGS.extend(["--preview-method", "auto"])

COMFY_LOG = "/content/comfyui_boot.log"
TUNNEL_LOG = "/content/cloudflared.log"
PORT = 8188

def port_open(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        return s.connect_ex(("127.0.0.1", port)) == 0
    finally:
        s.close()

# 0. Auto-heal runtime deps
print("[SYSTEM] Auto-healing runtime dependencies...")
os.system("pip install -q gguf piexif > /dev/null 2>&1")
os.system("pip cache purge > /dev/null 2>&1")

# 1. Cloudflared
print("[SYSTEM] Verifying Cloudflared...")
if not os.path.exists("/usr/bin/cloudflared"):
    os.system("apt-get install -y aria2 > /dev/null 2>&1")
    os.system("aria2c -x 16 -s 16 -k 1M https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -d /content -o cloudflared.deb > /dev/null 2>&1")
    os.system("dpkg -i /content/cloudflared.deb > /dev/null 2>&1")

# 2. Boot ComfyUI -> log file (process survives cell return)
if not os.path.exists(os.path.join(WORKSPACE, "main.py")):
    raise SystemExit("[FATAL] ComfyUI directory missing — re-run cell 1 (System Init).")

if port_open(PORT):
    print(f"[SYSTEM] ComfyUI already listening on :{PORT} — skipping boot.")
else:
    os.chdir(WORKSPACE)
    print(f"[SYSTEM] Booting ComfyUI [{MEMORY_PROFILE}] ...")
    cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", str(PORT)] + ARGS
    subprocess.Popen(cmd, stdout=open(COMFY_LOG, "wb"), stderr=subprocess.STDOUT)
    print(f"[SYSTEM] Waiting for local server :{PORT} (logs: {COMFY_LOG}) ...")
    for _ in range(150):
        if port_open(PORT):
            break
        time.sleep(1)
    else:
        tail = open(COMFY_LOG, errors="replace").read()[-2500:] if os.path.exists(COMFY_LOG) else "(no log)"
        raise SystemExit("[FATAL] Port 8188 did not open in 150s. Log tail:\n" + tail)
    print("[SYSTEM] Local server UP.")

# 3. Tunnel -> log file; poll for URL while streaming new ComfyUI log lines
if not os.path.exists(TUNNEL_LOG) or "trycloudflare.com" not in open(TUNNEL_LOG, errors="replace").read():
    print("[SYSTEM] Starting Cloudflare tunnel ...")
    subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                      "--edge-ip-version", "4", "--protocol", "http2"],
                     stdout=open(TUNNEL_LOG, "ab"), stderr=subprocess.STDOUT)

tunnel_url = None
comfy_pos = os.path.getsize(COMFY_LOG) if os.path.exists(COMFY_LOG) else 0
print("[SYSTEM] Streaming boot logs until tunnel is ready (usually 30-90s) ...")
for _ in range(120):
    time.sleep(1)
    if os.path.exists(COMFY_LOG):
        with open(COMFY_LOG, "r", errors="replace") as f:
            f.seek(comfy_pos)
            chunk = f.read()
            comfy_pos = f.tell()
        if chunk:
            sys.stdout.write(chunk)
            sys.stdout.flush()
    if os.path.exists(TUNNEL_LOG):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG, errors="replace").read())
        if m:
            tunnel_url = m.group(0)
            break

print()
if tunnel_url:
    print("=" * 60)
    print(f"ACTIVE TUNNEL: {tunnel_url}")
    print("=" * 60)
    print("[SYSTEM] Open the URL above in a new browser tab.")
    print("[SYSTEM] This cell is DONE — other cells can run now. Keep this tab open.")
else:
    print("[WARN] Tunnel URL not found after 120s. Processes keep running.")
    print(f"       Inspect: cat {TUNNEL_LOG} | tail -30")
    print("       Re-run this cell to retry the tunnel only.")


# **حفظ الكاش والنسخ الاحتياطي**

**في الخلية دي بنبني كاش سريع من العقد والإعدادات ونحفظه للجلسة الجاية حسب إعداد `CACHE_SOURCE` (على هافينج فيس و/أو جوجل درايف).**

**شغّلها قبل ما تقفل الجلسة عشان تاخد بالك من حفظ كل حاجة — الجلسة الجاية هتفتح في ثواني.**

In [ ]:
#@title 5. Build Tarball Cache & Backup
#@markdown Compresses custom_nodes + user into a tarball and saves it to the CACHE_SOURCE backend (hf and/or drive). Run before disconnecting.

import os, sys

CACHE_SOURCE = globals().get("CACHE_SOURCE", "hf")
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ComfyUI")
CACHE_TAR = globals().get("CACHE_TAR", os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar"))
HF_BOOT_CACHE_REPO = globals().get("HF_BOOT_CACHE_REPO", "awAyham/comfyui-boot-cache")

if not os.path.exists(LOCAL_WORKSPACE):
    print("[WARN] ComfyUI workspace not found. Nothing to backup.")
else:
    staging = "/content/comfy_ui_cache.tar"
    os.chdir(LOCAL_WORKSPACE)
    print("Compressing custom_nodes + user ...")
    rc = os.system(f'tar -cf "{staging}" custom_nodes user')
    if rc != 0:
        print("[WARN] tar returned non-zero — archive may be incomplete.")

    if CACHE_SOURCE in ("drive", "hf") and os.path.exists("/content/drive/MyDrive"):
        os.makedirs(os.path.dirname(CACHE_TAR), exist_ok=True)
        rc2 = os.system(f'cp "{staging}" "{CACHE_TAR}"')
        if rc2 == 0:
            print(f"[OK] Drive copy: {CACHE_TAR}")
        else:
            print(f"[WARN] Drive copy failed (rc={rc2}).")

    if CACHE_SOURCE == "hf":
        from huggingface_hub import HfApi
        size_gb = os.path.getsize(staging) / 1e9 if os.path.exists(staging) else 0
        print(f"[CACHE] Uploading {size_gb:.2f} GB to {HF_BOOT_CACHE_REPO} ...")
        HfApi().upload_file(path_or_fileobj=staging, path_in_repo="comfy_ui_cache.tar",
                            repo_id=HF_BOOT_CACHE_REPO, repo_type="model",
                            commit_message=f"Boot cache update ({size_gb:.2f} GB)")
        print("[OK] HF boot cache updated.")
    elif CACHE_SOURCE == "none":
        print("[CACHE] CACHE_SOURCE=none — backup skipped.")

    if os.path.exists(staging):
        os.remove(staging)
    print("[OK] Cache step finished. You may now safely disconnect.")